# Superstore Sales & Profitability Analysis

## 1. Project Objective & Dataset Source

**Objective.** Build an initial understanding of the Superstore dataset and assess its data
quality **before** making any cleaning decisions. This notebook is exploratory and strictly
read-only: it does not clean, correct, standardise, transform, export, or overwrite the source data.

**Dataset source.** Tableau's official sample dataset:
`https://public.tableau.com/app/sample-data/sample_-_superstore.xls`

**File location.** `data/raw/sample_-_superstore.xls` (relative to the repository), resolved
robustly at runtime so the notebook works whether Jupyter starts in the repository root or in the
`notebooks/` directory.

**Expected sheets:** `Orders`, `People`, `Returns`.

**Record-level terminology used throughout:**

- **Order-line record** — one row in `Orders` (a single product line within an order). `Orders` is line-level.
- **Order-level record** — one distinct `Order ID`. A single order may contain several line records.

Monetary fields (`Sales`, `Profit`) are recorded per order line, not per order.

## 2. Environment & Imports

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

print("Python executable:", sys.executable)
print("Python version:   ", sys.version.split()[0])
print("pandas version:   ", pd.__version__)
print("numpy version:    ", np.__version__)

Python executable: /home/abutaha/Github/salesAnalysis/.venv/bin/python
Python version:    3.14.8
pandas version:    3.0.6
numpy version:     2.5.4


## 3. Data Loading

In [2]:
def find_dataset_path(start_directory: Path, relative_path: str) -> Path:
    '''Return the dataset path, searching the start directory and its parents.'''
    for base_directory in [start_directory, *start_directory.parents]:
        candidate_path = base_directory / relative_path
        if candidate_path.exists():
            return candidate_path.resolve()
    raise FileNotFoundError(
        f"Could not find '{relative_path}' in '{start_directory}' or any parent directory."
    )


relative_dataset_path = "data/raw/sample_-_superstore.xls"
dataset_path = find_dataset_path(Path.cwd(), relative_dataset_path)

print("Notebook working directory:", Path.cwd())
print("Resolved dataset path:    ", dataset_path)

Notebook working directory: /home/abutaha/Github/salesAnalysis/notebooks
Resolved dataset path:     /home/abutaha/Github/salesAnalysis/data/raw/sample_-_superstore.xls


In [3]:
orders = pd.read_excel(dataset_path, sheet_name="Orders", engine="xlrd")
people = pd.read_excel(dataset_path, sheet_name="People", engine="xlrd")
returns = pd.read_excel(dataset_path, sheet_name="Returns", engine="xlrd")

sheets = {"Orders": orders, "People": people, "Returns": returns}

print("Loaded sheets:", list(sheets.keys()))
for sheet_name, sheet_data in sheets.items():
    print(f"  {sheet_name}: {sheet_data.shape[0]} rows x {sheet_data.shape[1]} columns")

Loaded sheets: ['Orders', 'People', 'Returns']
  Orders: 10194 rows x 21 columns
  People: 4 rows x 2 columns
  Returns: 296 rows x 2 columns


## 4. Initial Inspection

In [4]:
shape_summary = pd.DataFrame(
    {
        "Rows": [sheet_data.shape[0] for sheet_data in sheets.values()],
        "Columns": [sheet_data.shape[1] for sheet_data in sheets.values()],
    },
    index=list(sheets.keys()),
)
print("Sheet shapes:")
display(shape_summary)

Sheet shapes:


,Rows,Columns
Orders,10194,21
People,4,2
Returns,296,2


In [5]:
for sheet_name, sheet_data in sheets.items():
    print(f"[{sheet_name}] columns ({len(sheet_data.columns)}):")
    for position, column_name in enumerate(sheet_data.columns, start=1):
        print(f"  {position:2d}. {column_name}")
    print()

[Orders] columns (21):
   1. Row ID
   2. Order ID
   3. Order Date
   4. Ship Date
   5. Ship Mode
   6. Customer ID
   7. Customer Name
   8. Segment
   9. Country/Region
  10. City
  11. State/Province
  12. Postal Code
  13. Region
  14. Product ID
  15. Category
  16. Sub-Category
  17. Product Name
  18. Sales
  19. Quantity
  20. Discount
  21. Profit

[People] columns (2):
   1. Regional Manager
   2. Region

[Returns] columns (2):
   1. Order ID
   2. Returned



In [6]:
for sheet_name, sheet_data in sheets.items():
    print(f"[{sheet_name}] data types:")
    print(sheet_data.dtypes.to_string())
    print()

[Orders] data types:
Row ID                     int64
Order ID                     str
Order Date        datetime64[us]
Ship Date         datetime64[us]
Ship Mode                    str
Customer ID                  str
Customer Name                str
Segment                      str
Country/Region               str
City                         str
State/Province               str
Postal Code               object
Region                       str
Product ID                   str
Category                     str
Sub-Category                 str
Product Name                 str
Sales                    float64
Quantity                   int64
Discount                 float64
Profit                   float64

[People] data types:
Regional Manager    str
Region              str

[Returns] data types:
Order ID    str
Returned    str



In [7]:
for sheet_name, sheet_data in sheets.items():
    print(f"[{sheet_name}] first 5 rows:")
    display(sheet_data.head())

[Orders] first 5 rows:


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,US-2023-103800,2023-01-03,2023-01-07,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,...,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448,2,0.2,5.5512
1,2,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540,2,0.8,-5.4870
2,3,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.2,4.2717
3,4,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736,3,0.2,-64.7748
4,5,US-2023-141817,2023-01-05,2023-01-12,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,...,19143,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,19.536,3,0.2,4.8840


[People] first 5 rows:


,Regional Manager,Region
0,Sadie Pawthorne,West
1,Chuck Magee,East
2,Roxanne Rodriguez,Central
3,Fred Suzuki,South


[Returns] first 5 rows:


,Order ID,Returned
0,US-2023-100762,Yes
1,US-2023-100867,Yes
2,US-2023-102652,Yes
3,US-2023-103373,Yes
4,US-2023-103744,Yes


In [8]:
print("[Orders] descriptive statistics for numeric columns:")
display(orders.describe(include=[np.number]).T)

[Orders] descriptive statistics for numeric columns:


,count,mean,std,min,25%,50%,75%,max
Row ID,10194.0,5097.500000,2942.898656,1.000,2549.2500,5097.50,7645.750000,10194.000
Sales,10194.0,228.225854,619.906839,0.444,17.2200,53.91,209.500000,22638.480
Quantity,10194.0,3.791838,2.228317,1.000,2.0000,3.00,5.000000,14.000
Discount,10194.0,0.155385,0.206249,0.000,0.0000,0.20,0.200000,0.800
Profit,10194.0,28.673417,232.465115,-6599.978,1.7608,8.69,29.297925,8399.976


In [9]:
print("[Orders] descriptive statistics for non-numeric columns:")
display(orders.describe(exclude=[np.number]).T)

[Orders] descriptive statistics for non-numeric columns:


,count,unique,top,freq,mean,min,25%,50%,75%,max
Order ID,10194,5111,US-2026-100111,14,NaN,NaN,NaN,NaN,NaN,NaN
Order Date,10194,NaN,NaN,NaN,2025-04-29 11:48:25.002943,2023-01-03 00:00:00,2024-05-14 00:00:00,2025-06-25 00:00:00,2026-05-14 00:00:00,2026-12-30 00:00:00
Ship Date,10194,NaN,NaN,NaN,2025-05-03 10:52:45.626839,2023-01-07 00:00:00,2024-05-19 00:00:00,2025-06-28 00:00:00,2026-05-18 00:00:00,2027-01-05 00:00:00
Ship Mode,10194,4,Standard Class,6120,NaN,NaN,NaN,NaN,NaN,NaN
Customer ID,10194,804,WB-21850,41,NaN,NaN,NaN,NaN,NaN,NaN
Customer Name,10194,800,William Brown,41,NaN,NaN,NaN,NaN,NaN,NaN
Segment,10194,3,Consumer,5281,NaN,NaN,NaN,NaN,NaN,NaN
Country/Region,10194,2,United States,9994,NaN,NaN,NaN,NaN,NaN,NaN
City,10194,542,New York City,915,NaN,NaN,NaN,NaN,NaN,NaN
State/Province,10194,59,California,2001,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Data Quality Assessment

### 5.1 Missing values

In [10]:
for sheet_name, sheet_data in sheets.items():
    missing_counts = sheet_data.isna().sum()
    columns_with_missing = missing_counts[missing_counts > 0]
    print(f"[{sheet_name}] total missing cells: {int(missing_counts.sum())}")
    if columns_with_missing.empty:
        print("  no missing values in any column")
    else:
        print(columns_with_missing.to_string())
    print()

[Orders] total missing cells: 0
  no missing values in any column

[People] total missing cells: 0
  no missing values in any column

[Returns] total missing cells: 0
  no missing values in any column



### 5.2 Exact duplicate rows and duplicate business keys

In [11]:
print("=== Exact duplicate rows (identical across all columns) ===")
for sheet_name, sheet_data in sheets.items():
    print(f"[{sheet_name}] exact duplicate rows: {int(sheet_data.duplicated().sum())}")

print()
print("=== Orders: candidate business keys ===")
candidate_keys = {
    "Row ID": ["Row ID"],
    "Order ID": ["Order ID"],
    "Order ID + Product ID": ["Order ID", "Product ID"],
}
for label, key_columns in candidate_keys.items():
    extra_duplicate_rows = int(orders.duplicated(subset=key_columns).sum())
    distinct_keys = orders[key_columns].drop_duplicates().shape[0]
    print(
        f"{label:24s} -> distinct keys: {distinct_keys:6d} | "
        f"extra duplicate rows: {extra_duplicate_rows}"
    )

print()
print("=== Orders: duplicate records ignoring the unique 'Row ID' column ===")
business_columns = [column for column in orders.columns if column != "Row ID"]
duplicate_record_mask = orders.duplicated(subset=business_columns, keep=False)
print("Rows belonging to duplicate record groups:", int(duplicate_record_mask.sum()))
if duplicate_record_mask.any():
    display(
        orders.loc[duplicate_record_mask].sort_values(business_columns[:4])[
            ["Row ID", "Order ID", "Product ID", "Sales", "Quantity", "Discount", "Profit"]
        ]
    )

=== Exact duplicate rows (identical across all columns) ===


[Orders] exact duplicate rows: 0
[People] exact duplicate rows: 0
[Returns] exact duplicate rows: 0

=== Orders: candidate business keys ===
Row ID                   -> distinct keys:  10194 | extra duplicate rows: 0
Order ID                 -> distinct keys:   5111 | extra duplicate rows: 5083


Order ID + Product ID    -> distinct keys:  10183 | extra duplicate rows: 11

=== Orders: duplicate records ignoring the unique 'Row ID' column ===


Rows belonging to duplicate record groups: 4


,Row ID,Order ID,Product ID,Sales,Quantity,Discount,Profit
1698,1699,CA-2023-153623,FUR-FU-10002501,99.120,8,0.0,35.4144
1699,1700,CA-2023-153623,FUR-FU-10002501,99.120,8,0.0,35.4144
390,391,US-2023-150119,FUR-CH-10002965,281.372,2,0.3,-12.0588
391,392,US-2023-150119,FUR-CH-10002965,281.372,2,0.3,-12.0588


### 5.3 Date ranges and future-dated orders

In [12]:
# Fixed project analysis date (2026-10-13); do not derive it from wall-clock time.
analysis_date = pd.Timestamp("2026-10-13")
print("Analysis date (fixed):", analysis_date.date())

order_dates = pd.to_datetime(orders["Order Date"])
ship_dates = pd.to_datetime(orders["Ship Date"])

print(f"Order Date range: {order_dates.min().date()} -> {order_dates.max().date()}")
print(f"Ship Date range:  {ship_dates.min().date()} -> {ship_dates.max().date()}")

future_order_mask = order_dates > analysis_date
future_orders = orders.loc[future_order_mask]
print(
    f"Future-dated order lines (Order Date > analysis date): {future_orders.shape[0]} "
    f"line records across {future_orders['Order ID'].nunique()} distinct Order IDs"
)

print("Rows where Ship Date is earlier than Order Date:", int((ship_dates < order_dates).sum()))
print("Order Date year distribution:")
print(order_dates.dt.year.value_counts().sort_index().to_string())

Analysis date (fixed): 2026-10-13
Order Date range: 2023-01-03 -> 2026-12-30
Ship Date range:  2023-01-07 -> 2027-01-05
Future-dated order lines (Order Date > analysis date): 1106 line records across 582 distinct Order IDs
Rows where Ship Date is earlier than Order Date: 0
Order Date year distribution:
Order Date
2023    2051
2024    2130
2025    2634
2026    3379


### 5.4 Numeric ranges and unusual values

In [13]:
numeric_columns = ["Sales", "Quantity", "Discount", "Profit"]
numeric_summary = orders[numeric_columns].describe().T
numeric_summary["negative_count"] = (orders[numeric_columns] < 0).sum()
numeric_summary["zero_count"] = (orders[numeric_columns] == 0).sum()
display(numeric_summary)

print("Discount distinct values:", sorted(orders["Discount"].unique().tolist()))
print("Quantity distinct values:", sorted(orders["Quantity"].unique().tolist()))
print("Sales values below 0:", int((orders["Sales"] < 0).sum()))
print(
    "Discount values outside [0, 1]:",
    int(((orders["Discount"] < 0) | (orders["Discount"] > 1)).sum()),
)
print("Profit values below 0 (loss-making lines):", int((orders["Profit"] < 0).sum()))
print("Profit values equal to 0:", int((orders["Profit"] == 0).sum()))

,count,mean,std,min,25%,50%,75%,max,negative_count,zero_count
Sales,10194.0,228.225854,619.906839,0.444,17.2200,53.91,209.500000,22638.480,0,0
Quantity,10194.0,3.791838,2.228317,1.000,2.0000,3.00,5.000000,14.000,0,0
Discount,10194.0,0.155385,0.206249,0.000,0.0000,0.20,0.200000,0.800,0,4925
Profit,10194.0,28.673417,232.465115,-6599.978,1.7608,8.69,29.297925,8399.976,1901,65


Discount distinct values: [0.0, 0.1, 0.15, 0.2, 0.3, 0.32, 0.4, 0.45, 0.5, 0.6, 0.7, 0.8]
Quantity distinct values: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
Sales values below 0: 0
Discount values outside [0, 1]: 0
Profit values below 0 (loss-making lines): 1901
Profit values equal to 0: 65


### 5.5 Identifier-to-name consistency

In [14]:
print("=== Customer ID <-> Customer Name ===")
names_per_customer_id = orders.groupby("Customer ID")["Customer Name"].nunique()
customer_ids_per_name = orders.groupby("Customer Name")["Customer ID"].nunique()
print("Customer IDs mapping to more than one name:", int((names_per_customer_id > 1).sum()))
print("Customer Names mapping to more than one ID:", int((customer_ids_per_name > 1).sum()))
ambiguous_customer_names = customer_ids_per_name[customer_ids_per_name > 1]
if not ambiguous_customer_names.empty:
    display(ambiguous_customer_names)

print()
print("=== Product ID <-> Product Name ===")
names_per_product_id = orders.groupby("Product ID")["Product Name"].nunique()
product_ids_per_name = orders.groupby("Product Name")["Product ID"].nunique()
print("Product IDs mapping to more than one name:", int((names_per_product_id > 1).sum()))
print("Product Names mapping to more than one ID:", int((product_ids_per_name > 1).sum()))

=== Customer ID <-> Customer Name ===
Customer IDs mapping to more than one name: 0
Customer Names mapping to more than one ID: 1


Customer Name
Harry Olson    5
Name: Customer ID, dtype: int64


=== Product ID <-> Product Name ===
Product IDs mapping to more than one name: 32
Product Names mapping to more than one ID: 17


### 5.6 Postal Code type and format consistency

In [15]:
postal_code_python_types = orders["Postal Code"].map(lambda value: type(value).__name__)
postal_code_string_lengths = orders["Postal Code"].astype(str).str.len()

print("Postal Code Python value types:")
print(postal_code_python_types.value_counts().to_string())
print()
print("Postal Code string length counts:")
print(postal_code_string_lengths.value_counts().sort_index().to_string())
print()
print("Postal Code string length by Country/Region:")
display(pd.crosstab(orders["Country/Region"], postal_code_string_lengths))

is_numeric_postal_code = orders["Postal Code"].map(
    lambda value: isinstance(value, (int, float))
)
print()
print("Numeric Postal Code values:", int(is_numeric_postal_code.sum()))
print("Text (string) Postal Code values:", int((~is_numeric_postal_code).sum()))
print(
    "Distinct text Postal Code values:",
    sorted(orders.loc[~is_numeric_postal_code, "Postal Code"].astype(str).unique().tolist()),
)
print()

united_states_four_character = orders.loc[
    (orders["Country/Region"] == "United States") & (postal_code_string_lengths == 4)
]
print("United States Postal Codes with 4 characters:", united_states_four_character.shape[0])
print()

print("Example Postal Code values by Country/Region:")
for country_name in sorted(orders["Country/Region"].dropna().unique()):
    example_values = (
        orders.loc[orders["Country/Region"] == country_name, "Postal Code"]
        .astype(str)
        .drop_duplicates()
        .head(5)
        .tolist()
    )
    print(f"  {country_name}: {example_values}")

Postal Code Python value types:
Postal Code
int    9983
str     211

Postal Code string length counts:
Postal Code
3     200
4     438
5    9556

Postal Code string length by Country/Region:


Postal Code,3,4,5
Country/Region,,,
Canada,200,0,0
United States,0,438,9556



Numeric Postal Code values: 9983
Text (string) Postal Code values: 211
Distinct text Postal Code values: ['05401', 'A0A', 'B3H', 'C0A', 'E1A', 'G1B', 'H1A', 'H1B', 'H1C', 'M2N', 'M3C', 'M5H', 'M7A', 'R0H', 'R3R', 'S0G', 'T1Y', 'T2C', 'T5A', 'V5K', 'V6B', 'V6E', 'V6G', 'V6Z']



United States Postal Codes with 4 characters: 438

Example Postal Code values by Country/Region:


  Canada: ['M7A', 'T2C', 'M3C', 'V6Z', 'H1A']
  United States: ['77095', '60540', '19143', '42420', '30605']


## 6. Relationship Validation

### 6.1 Orders to Returns

In [16]:
order_ids = set(orders["Order ID"].unique())
return_order_ids = set(returns["Order ID"].unique())

matched_return_order_ids = return_order_ids & order_ids
unmatched_return_order_ids = return_order_ids - order_ids

print("Distinct Order IDs in Orders:  ", len(order_ids))
print("Distinct Order IDs in Returns: ", len(return_order_ids))
print("Return Order IDs also present in Orders:", len(matched_return_order_ids))
print("Return Order IDs not found in Orders:  ", len(unmatched_return_order_ids))
print("Duplicate Order IDs within Returns:", int(returns["Order ID"].duplicated().sum()))

return_rate = len(matched_return_order_ids) / len(order_ids)
print(f"Return rate (distinct returned orders / distinct orders): {return_rate:.2%}")

Distinct Order IDs in Orders:   5111
Distinct Order IDs in Returns:  296
Return Order IDs also present in Orders: 296
Return Order IDs not found in Orders:   0
Duplicate Order IDs within Returns: 0
Return rate (distinct returned orders / distinct orders): 5.79%


### 6.2 Orders to People region coverage

In [17]:
orders_regions = set(orders["Region"].dropna().unique())
people_regions = set(people["Region"].dropna().unique())

print("Regions in Orders:", sorted(orders_regions))
print("Regions in People:", sorted(people_regions))
print("Regions in Orders but missing from People:", sorted(orders_regions - people_regions))
print("Regions in People but not in Orders:     ", sorted(people_regions - orders_regions))

Regions in Orders: ['Central', 'East', 'South', 'West']
Regions in People: ['Central', 'East', 'South', 'West']
Regions in Orders but missing from People: []
Regions in People but not in Orders:      []


### 6.3 Order-level vs order-line records

In [18]:
order_line_records = orders.shape[0]
order_level_records = orders["Order ID"].nunique()
lines_per_order = orders.groupby("Order ID").size()

print("Order-line records (rows in Orders):     ", order_line_records)
print("Order-level records (distinct Order IDs):", order_level_records)
print(f"Average line records per order: {lines_per_order.mean():.2f}")
print("Minimum lines per order:", int(lines_per_order.min()))
print("Maximum lines per order:", int(lines_per_order.max()))
print()
print("Line-records-per-order distribution:")
print(lines_per_order.value_counts().sort_index().to_string())

Order-line records (rows in Orders):      10194
Order-level records (distinct Order IDs): 5111
Average line records per order: 1.99
Minimum lines per order: 1
Maximum lines per order: 14

Line-records-per-order distribution:
1     2593
2     1242
3      612
4      344
5      164
6       72
7       51
8       16
9       10
10       3
11       2
12       1
14       1


## 7. Findings & Cleaning Decisions

In [19]:
summary_metrics = {
    "orders_line_records": orders.shape[0],
    "orders_distinct_order_ids": orders["Order ID"].nunique(),
    "people_rows": people.shape[0],
    "returns_rows": returns.shape[0],
    "returns_distinct_order_ids": returns["Order ID"].nunique(),
    "returns_unmatched_order_ids": len(unmatched_return_order_ids),
    "return_rate_distinct_orders": round(len(matched_return_order_ids) / len(order_ids), 4),
    "orders_regions_not_in_people": sorted(orders_regions - people_regions),
    "total_missing_cells_all_sheets": int(
        sum(sheet_data.isna().sum().sum() for sheet_data in sheets.values())
    ),
    "exact_duplicate_rows_all_sheets": int(
        sum(sheet_data.duplicated().sum() for sheet_data in sheets.values())
    ),
    "future_dated_order_lines_vs_analysis_date": int(future_order_mask.sum()),
}

print("Summary metrics:")
for metric_name, metric_value in summary_metrics.items():
    print(f"  {metric_name}: {metric_value}")

Summary metrics:


  orders_line_records: 10194
  orders_distinct_order_ids: 5111
  people_rows: 4
  returns_rows: 296
  returns_distinct_order_ids: 296
  returns_unmatched_order_ids: 0
  return_rate_distinct_orders: 0.0579
  orders_regions_not_in_people: []
  total_missing_cells_all_sheets: 0
  exact_duplicate_rows_all_sheets: 0
  future_dated_order_lines_vs_analysis_date: 1106


> This section documents **observations and recommended follow-up decisions only**. No row is
> deleted, corrected, standardised, or overwritten in this notebook. All numbers below are
> produced by the cells above at runtime.

### Confirmed observations

1. **Structure.** `Orders` is line-level: 10,194 rows for 5,111 distinct `Order ID`s (about 1.99
   lines per order, 1 to 14 lines per order). `People` has 4 rows and `Returns` has 296 rows. The
   reported sheet shapes match the expected counts (Orders 10,194; People 4; Returns 296).
2. **Missing values.** Zero missing cells across all three sheets.
3. **Exact duplicates.** Zero rows are identical across all columns in any sheet.
4. **Duplicate business keys.** `Row ID` is a unique, contiguous key. `Order ID` is not unique
   (expected, because of multiple lines). `Order ID + Product ID` is nearly unique but has a small
   number of extra duplicate rows. A handful of row pairs are identical once `Row ID` is ignored.
5. **Dates.** `Order Date` and `Ship Date` parse as datetimes. The range spans several years, and a
   number of order lines are dated **after the fixed analysis date** (see section 5.3). No rows
   have a `Ship Date` earlier than their `Order Date`.
6. **Numeric fields.** `Sales` are non-negative; `Profit` includes both losses and zeros (expected
   for a profitability dataset); `Quantity` is integer-valued within a small range; `Discount`
   lies within `[0, 1]` but includes a few values that are not multiples of 0.1 (for example 0.15,
   0.32, 0.45).
7. **Identifiers vs names.** `Customer ID` is 1:1 with `Customer Name` in one direction, but one
   customer name maps to multiple customer IDs. `Product ID` and `Product Name` are not strictly
   1:1 in either direction.
8. **Returns.** Every `Order ID` in `Returns` also exists in `Orders`; no return order ID is
   duplicated. The return rate is computed from **distinct order IDs** (distinct returned orders
   divided by distinct orders).
9. **People.** The four regions in `People` exactly match the four regions in `Orders`; there are
   no missing or extra regions.
10. **Mixed-type field.** `Postal Code` stores a mix of Python `int` (9,983) and `str` (211)
    values. All Canadian codes are 3-character strings (200 rows). In the United States, 9,556
    values are five characters long and 438 are four characters long. Eleven United States rows
    store the code as the five-character string `05401`, while the remaining five-character
    United States values are numeric. Because the 438 four-character values are stored as
    numbers, no leading zero is present in the data; whether any originally had a leading zero
    cannot be determined from this dataset alone. These values should be reviewed before any
    geographic analysis (see section 5.6).

### Recommended follow-up decisions (not executed here)

- Confirm with the data owner whether the identical rows found in section 5.2 are genuine
  duplicates before deciding whether to de-duplicate them.
- Define and document the analysis grain: aggregate to order level before computing any
  order-level metric, since `Sales` and `Profit` are recorded per line.
- Investigate the future-dated orders to determine whether they are intentional; do not assume
  they are errors.
- Decide how to treat `Postal Code` (preserve as text) and review the 438 four-character United
  States values and the mixed numeric/text storage; do not assume a leading zero was dropped.
- Decide on a canonical `Product ID` to `Product Name` mapping before joining product data across
  sources.
- Decide whether the duplicate customer name represents a data-entry issue or legitimately
  distinct customers.
- Keep `Returns` keyed by distinct `Order ID` and state the return-rate denominator explicitly in
  later reporting.